# GRACE H1 — Kaggle GPU training

Bắt đầu từ notebooks/full-pipeline.ipynb ở local, đến Bước 1 thì chuyển sang notebook này. Nhánh reveal: đính kèm training-inputs.zip mới có datasets=[reveal], dùng Kaggle session mới. Bật GPU và Internet; đính kèm grace-h1-training-inputs.zip làm Kaggle Dataset. Chỉ train head từ embedding cache; không gọi LLM, không gửi test code/labels vào GPU job.

In [ ]:
from pathlib import Path,PurePosixPath
from contextlib import ExitStack
import json,zipfile,hashlib,shutil,os,sys,subprocess
ROOT=Path('/kaggle/working/GRACE-H1-SecurityProjection-reveal')
zips=list(Path('/kaggle/input').rglob('grace-h1-training-inputs.zip'))
expanded=[]
for p in Path('/kaggle/input').rglob('bundle_manifest.json'):
    try:
        if json.loads(p.read_text())['kind']=='training_inputs': expanded.append(p.parent)
    except (ValueError,KeyError): pass
options=zips if zips else expanded
BUNDLE_PATH=Path(os.environ['GRACE_H1_BUNDLE']) if os.environ.get('GRACE_H1_BUNDLE') else (options[0] if len(options)==1 else None)
if BUNDLE_PATH is None: raise RuntimeError('Attach one training bundle (zip or expanded) or set GRACE_H1_BUNDLE')
ROOT.mkdir(parents=True,exist_ok=True)
with ExitStack() as stack:
    if BUNDLE_PATH.is_file() and BUNDLE_PATH.suffix=='.zip':
        z=stack.enter_context(zipfile.ZipFile(BUNDLE_PATH))
        manifest=json.loads(z.read('bundle_manifest.json'))
        open_member=lambda name:z.open(name)
        archive_names=z.namelist()
    else:
        folder=BUNDLE_PATH if BUNDLE_PATH.is_dir() else BUNDLE_PATH.parent
        manifest=json.loads((folder/'bundle_manifest.json').read_text())
        open_member=lambda name:(folder/name).open('rb')
        archive_names=None
    if manifest['schema']!='grace_h1_bundle_v1' or manifest['kind']!='training_inputs': raise ValueError('Wrong bundle')
    names=[e['path'] for e in manifest['entries']]
    if len(set(n.casefold() for n in names))!=len(names): raise ValueError('Duplicate paths')
    if archive_names is not None and sorted(archive_names)!=sorted(names+['bundle_manifest.json']): raise ValueError('Unexpected members')
    for e in manifest['entries']:
        name=e['path'];p=PurePosixPath(name)
        if p.is_absolute() or '..' in p.parts or ':' in name or '\\' in name: raise ValueError('Unsafe path')
        target=ROOT/name;target.resolve().relative_to(ROOT)
        target.parent.mkdir(parents=True,exist_ok=True)
        temp=target.with_name(target.name+'.bundle.tmp')
        h=hashlib.sha256();size=0
        with open_member(name) as src,temp.open('wb') as dst:
            while True:
                chunk=src.read(1024*1024)
                if not chunk: break
                h.update(chunk);size+=len(chunk);dst.write(chunk)
        if size!=e['bytes'] or h.hexdigest()!=e['sha256']: raise ValueError('Bundle checksum mismatch')
        os.replace(temp,target)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
print(ROOT,'extracted and checksummed')

In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements.txt')])
import torch
print('torch:',torch.__version__,'CUDA:',torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

In [ ]:
from grace_h1.io import read_json
from grace_h1.experiment import train_all
from grace_h1.portable import create_bundle
config=read_json(ROOT/'configs'/'experiment.json')
# Keep dimension/batch/seed/objective identical to local for exact head reuse.
if config['datasets'] != ['reveal']: raise ValueError('Attach the new ReVeal training-inputs bundle')
print(json.dumps(config['training'],indent=2))
selection=train_all(ROOT,config)
for dataset,heads in selection['heads'].items():
    for objective,h in heads.items():
        print(dataset,objective,'best validation MCC:',h['best']['metrics']['mcc'],
              'epoch:',h['best']['epoch'],'parameters:',h['parameters'])

In [ ]:
destination=Path('/kaggle/working/grace-h1-trained-heads.zip')
print(create_bundle(ROOT,destination,'trained_heads'))
from IPython.display import FileLink,display
display(FileLink(str(destination)))

## Quay lại notebook local

Tải grace-h1-trained-heads.zip về bundles/ của project local. Mở full-pipeline.ipynb và Run All lại; Bước 1 sẽ import/reuse head khớp.

Đến Bước 2, notebook local hướng dẫn sang colab_cpu_prepare.ipynb trên Colab CPU. Mang cùng trained-heads.zip sang Colab, kèm cpu-inputs.zip và joern-runtime.zip. Colab prepare xong mới quay local để gọi LLM.